# SoninkeGPT — lot audio de 100 mots soninké–français

Ce notebook prépare les audios soninké du lot **LOT-100-2026-10-25** avec la voix de référence du projet.

- La liste contient 100 mots et leurs traductions françaises reprises du lexique source.
- Les traductions et les graphies ne sont jamais réécrites par ce notebook.
- Les WAV générés sont des candidats à écouter : ils ne sont **pas** déclarés validés automatiquement.
- Les fichiers existants et le manifeste de progression sont conservés dans Google Drive pour permettre la reprise.
- Sélectionner un environnement d'exécution GPU dans Colab si disponible. Le modèle peut être lent ou ne pas confirmer la prise en charge du soninké ; dans ce cas, le notebook s'arrête sans remplacer les mots par une autre langue.
- Vérifier les conditions de licence d'OmniVoice avant tout usage commercial.

In [ ]:
%pip -q install omnivoice soundfile pandas numpy

In [ ]:
from pathlib import Path
from google.colab import drive, files
from IPython.display import Audio, display
import pandas as pd
import numpy as np
import soundfile as sf
import hashlib, json, urllib.request, zipfile, shutil, time, os
import torch

drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/SONINKE_TTS_LIBRE')
BATCH = PROJECT / 'audio-batches' / 'LOT-100-2026-10-25'
AUDIO_DIR = BATCH / 'audio' / 'generated'
BATCH.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

REPO = 'https://raw.githubusercontent.com/aydiarra-star/soninkeGPT/main'
QUEUE_URL = REPO + '/data/source/audio-batch-100-2026-10-25.csv'
REF_A_URL = REPO + '/audio/snk-00054.wav'
REF_B_URL = REPO + '/audio/snk-00258.wav'

queue_path = BATCH / 'audio-batch-100-2026-10-25.csv'
if not queue_path.exists():
    urllib.request.urlretrieve(QUEUE_URL, queue_path)
queue = pd.read_csv(queue_path, keep_default_na=False, dtype=str)
assert len(queue) == 100, f'Le lot doit contenir exactement 100 lignes, reçu : {len(queue)}'
assert queue['soninke'].str.strip().ne('').all(), 'Au moins une forme soninké est vide.'
assert queue['francais'].str.strip().ne('').all(), 'Au moins une traduction française est vide.'
assert queue['soninke'].duplicated().sum() == 0, 'Doublons exacts dans le lot : arrêt pour vérification.'
print(f'Lot chargé : {len(queue)} mots ; {queue["francais"].str.strip().ne("").sum()} traductions présentes.')
display(queue[['id','soninke','francais','generation_status']].head(10))

In [ ]:
ref_a = BATCH / 'reference-snk-00054.wav'
ref_b = BATCH / 'reference-snk-00258.wav'
for url, path in [(REF_A_URL, ref_a), (REF_B_URL, ref_b)]:
    if not path.exists():
        urllib.request.urlretrieve(url, path)
for path in [ref_a, ref_b]:
    info = sf.info(path)
    assert info.frames > 0 and info.samplerate > 0, f'Référence audio invalide : {path}'
a, sr_a = sf.read(ref_a)
b, sr_b = sf.read(ref_b)
assert sr_a == sr_b, 'Les deux références n’ont pas la même fréquence d’échantillonnage.'
if a.ndim > 1: a = a.mean(axis=1)
if b.ndim > 1: b = b.mean(axis=1)
reference_path = BATCH / 'reference-combinee.wav'
sf.write(reference_path, np.concatenate([a, np.zeros(int(0.4*sr_a)), b]), sr_a)
print('Références vocales vérifiées et préparées.')

In [ ]:
from omnivoice import OmniVoice

if torch.cuda.is_available():
    DEVICE = 'cuda:0'
    DTYPE = torch.float16
else:
    DEVICE = 'cpu'
    DTYPE = torch.float32
print('Périphérique :', DEVICE, '| GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun GPU détecté')
if DEVICE == 'cpu':
    print('ATTENTION : le lot de 100 mots peut être très lent sur CPU. Si possible, activez un GPU dans Exécution > Modifier le type d’exécution.')

model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=DEVICE, dtype=DTYPE)
language_names = {str(x).casefold() for x in model.supported_language_names()}
language_ids = {str(x).casefold() for x in model.supported_language_ids()}
print('Langues déclarées par le modèle contenant « sonink » :', sorted(x for x in language_names if 'sonink' in x))
if not (any('sonink' in x for x in language_names) or 'snk' in language_ids):
    raise RuntimeError('Le modèle ne confirme pas la prise en charge du soninké. Arrêt sans génération audio de substitution.')

REF_TEXT = 'Daga goli. Li taxu.'

In [ ]:
# Reprise sûre : chaque WAV est contrôlé avant d’être marqué comme généré.
# Le manifeste est sauvegardé après chaque mot. Les traductions d’origine restent inchangées.
manifest_path = BATCH / 'audio-batch-100-progress.csv'
if manifest_path.exists():
    progress = pd.read_csv(manifest_path, keep_default_na=False, dtype=str)
    if len(progress) != 100 or progress['id'].tolist() != queue['id'].tolist():
        raise RuntimeError('Le manifeste existant ne correspond pas exactement au lot de 100 ; arrêt pour éviter toute perte.')
    # On ne remplace jamais les traductions conservées dans la file initiale.
    if progress['francais'].tolist() != queue['francais'].tolist() or progress['soninke'].tolist() != queue['soninke'].tolist():
        raise RuntimeError('Les formes ou traductions du manifeste ont changé ; arrêt pour préserver les données.')
else:
    progress = queue.copy()
    progress['generation_status'] = 'en_attente_generation'
    progress['pronunciation_validation'] = 'en_attente_ecoute_referent'
    progress['audio_file_created'] = ''
    progress['error'] = ''
    progress.to_csv(manifest_path, index=False, encoding='utf-8-sig')

def valid_wav(path):
    try:
        info = sf.info(path)
        return info.frames > 0 and info.samplerate > 0 and info.channels > 0
    except Exception:
        return False

success = 0
errors = 0
for idx, row in queue.iterrows():
    text = row['soninke']
    # Chemin déterministe compatible avec data/voice-memory.json
    digest = hashlib.sha256(text.encode('utf-8')).hexdigest()[:24]
    filename = f'{digest}.wav'
    out_path = AUDIO_DIR / filename
    old_status = progress.at[idx, 'generation_status']
    if valid_wav(out_path):
        progress.at[idx, 'generation_status'] = 'wav_present_controle_technique_ok'
        progress.at[idx, 'audio_file_created'] = f'audio/generated/{filename}'
        progress.at[idx, 'error'] = ''
        success += 1
        continue
    progress.at[idx, 'generation_status'] = 'generation_en_cours'
    progress.to_csv(manifest_path, index=False, encoding='utf-8-sig')
    try:
        result = model.generate(
            text=text,
            language='snk',
            ref_audio=str(reference_path),
            ref_text=REF_TEXT,
            num_step=32
        )
        samples = np.asarray(result[0]).squeeze()
        if samples.ndim != 1 or samples.size == 0 or not np.isfinite(samples).all():
            raise ValueError('Sortie audio vide ou invalide')
        tmp = out_path.with_suffix('.tmp.wav')
        sf.write(tmp, samples, int(model.sampling_rate), subtype='PCM_16')
        if not valid_wav(tmp):
            tmp.unlink(missing_ok=True)
            raise ValueError('Le WAV créé n’a pas passé le contrôle technique')
        tmp.replace(out_path)
        progress.at[idx, 'generation_status'] = 'wav_genere_a_ecouter'
        progress.at[idx, 'audio_file_created'] = f'audio/generated/{filename}'
        progress.at[idx, 'pronunciation_validation'] = 'en_attente_ecoute_referent'
        progress.at[idx, 'error'] = ''
        success += 1
        print(f'[{idx+1}/100] WAV créé : {text} — {row["francais"]}')
    except Exception as exc:
        progress.at[idx, 'generation_status'] = 'echec_generation'
        progress.at[idx, 'error'] = str(exc)
        errors += 1
        print(f'[{idx+1}/100] ÉCHEC : {text} — {exc}')
    progress.to_csv(manifest_path, index=False, encoding='utf-8-sig')

print(f'Fin du passage : {success} WAV présents/produits ; {errors} échecs. Les échecs sont conservés dans le manifeste.')

In [ ]:
# Vérification finale : les 100 graphies et traductions doivent toujours être présentes.
final = pd.read_csv(manifest_path, keep_default_na=False, dtype=str)
assert len(final) == 100
assert final['soninke'].tolist() == queue['soninke'].tolist()
assert final['francais'].tolist() == queue['francais'].tolist()
print(final['generation_status'].value_counts(dropna=False))
display(final[['id','soninke','francais','generation_status','pronunciation_validation','error']].head(20))

# ZIP : manifeste complet + tous les WAV présents. Aucun échec ni traduction n’est supprimé.
zip_path = BATCH / 'LOT-100-2026-10-25-audios-et-traductions.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(manifest_path, arcname='audio-batch-100-progress.csv')
    zf.write(queue_path, arcname='audio-batch-100-original.csv')
    for wav in sorted(AUDIO_DIR.glob('*.wav')):
        if valid_wav(wav):
            zf.write(wav, arcname='audio/generated/' + wav.name)
print('Archive prête :', zip_path, '| taille (Mo) :', round(zip_path.stat().st_size/1e6, 2))
files.download(str(zip_path))